# Table of contents

1. [Load Packages](#Load-Packages)
2. Import Libraries
3. Download NLTK Resources
4. Initialize Preprocessing Tools
5. Load the Data
6. Explore Data Structure
7. Data Cleaning
   - Remove Nulls and Duplicates
   - Clean Text Columns
8. Check for Duplicates and Missing Values
9. Validate URLs
10. Standardize & Group Categories
11. Encode Labels
12. Vectorize Text
13. Split Data for Training and Validation
14. Model Training
    - Logistic Regression
    - Random Forest
    - Multinomial Naive Bayes
    - K-Nearest Neighbors
15. Model Evaluation
    - Confusion Matrix
    - Classification Report
    - Radar Chart Comparison
16. Select Best Performing Model
17. Save Model and Components
    - Pickle Files
    - Joblib Files
18. Final Deployment
    - Load Model
    - Make Predictions


# Load Packages

In [ ]:
!pip install nltk
!pip install contractions

Running !pip install nltk and !pip install contractions installs the nltk library for natural language processing tasks and the contractions library for expanding text contractions, ensuring the necessary tools are available for your NLP project.

In [ ]:
iimport pandas as pd
import numpy as np
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import CountVectorizer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk
import contractions
from bs4 import BeautifulSoup
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

This code imports various libraries and modules necessary for data manipulation, preprocessing, and machine learning tasks. It includes tools for handling data (pandas, numpy), text processing (re, string, nltk, contractions, BeautifulSoup), splitting datasets (train_test_split), encoding labels (LabelEncoder), vectorizing text (CountVectorizer), and building machine learning models (LogisticRegression, RandomForestClassifier, MultinomialNB, KNeighborsClassifier). Additionally, it imports modules for evaluating model performance (confusion_matrix, classification_report) and visualizing data (matplotlib, seaborn).

# Loading the data and text preprossing

In [41]:
# Download NLTK resources (only run once)
import nltk
nltk.download(['punkt','stopwords'])
nltk.download('wordnet')

[nltk_data] <urlopen error [WinError 10060] A connection attempt
[nltk_data]     failed because the connected party did not properly
[nltk_data]     respond after a period of time, or established
[nltk_data]     connection failed because connected host has failed to
[nltk_data]     respond>
[nltk_data] Error loading wordnet: <urlopen error [WinError 10060] A
[nltk_data]     connection attempt failed because the connected party
[nltk_data]     did not properly respond after a period of time, or
[nltk_data]     established connection failed because connected host
[nltk_data]     has failed to respond>


False

Running nltk.download('stopwords') and nltk.download('wordnet') downloads the NLTK resources for stopwords and the WordNet lexical database, which are essential for text preprocessing tasks like removing common words and lemmatization. This step ensures these resources are available for your NLP project.

In [ ]:
# Initialize tools
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

This code initializes tools for text preprocessing by creating a set of English stopwords to filter out common words and setting up a lemmatizer to reduce words to their base forms, aiding in text normalization for NLP tasks.

In [10]:
# Load data
train=pd.read_csv('train.csv')
test=pd.read_csv('test.csv')

This code loads two CSV files named train.csv and test.csv into pandas DataFrames called train and test for data analysis or machine learning tasks.

In [11]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5520 entries, 0 to 5519
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   headlines    5520 non-null   object
 1   description  5520 non-null   object
 2   content      5520 non-null   object
 3   url          5520 non-null   object
 4   category     5520 non-null   object
dtypes: object(5)
memory usage: 215.8+ KB


In [12]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   headlines    2000 non-null   object
 1   description  2000 non-null   object
 2   content      2000 non-null   object
 3   url          2000 non-null   object
 4   category     2000 non-null   object
dtypes: object(5)
memory usage: 78.2+ KB


# Data Processing

In [24]:
train['full_text'] = train['headlines'].fillna('') + ' ' + train['description'].fillna('') + ' ' + train['content'].fillna('')
test['full_text'] = test['headlines'].fillna('') + ' ' + test['description'].fillna('') + ' ' + test['content'].fillna('')

This code combines the headlines, description, and content fields into a single full_text field for both the training and testing datasets, ensuring that all relevant text information is consolidated into one column for easier processing and analysis.

In [25]:
# Step 1: Import necessary libraries
import pandas as pd
import string
import os

# Define text cleaning function
def clean_text(text):
    if isinstance(text, str):
        text = text.lower()  # Convert to lowercase
        text = text.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation
        text = text.strip()  # Remove leading/trailing whitespace
    return text

#  Drop rows with null values
train.dropna(inplace=True)
test.dropna(inplace=True)

# Remove duplicate rows
train.drop_duplicates(inplace=True)
test.drop_duplicates(inplace=True)

# Apply cleaning to text columns
for col in train.select_dtypes(include='object').columns:
    train[col] = train[col].apply(clean_text)

for col in test.select_dtypes(include='object').columns:
    test[col] = test[col].apply(clean_text)

This code defines a function to clean text by converting it to lowercase, removing punctuation, and stripping whitespace. It then drops rows with null values and removes duplicate rows from the train and test DataFrames. Finally, it applies the cleaning function to all text columns in these DataFrames, ensuring the text data is preprocessed and ready for analysis or model training.

In [26]:
print(train.columns.tolist())
print(test.columns.tolist())

['headlines', 'description', 'content', 'url', 'category', 'article_length', 'full_text']
['headlines', 'description', 'content', 'url', 'category', 'full_text']


This code prints the list of column names in the train and test DataFrames, allowing you to see the structure and available fields in both the training and testing datasets.

In [27]:
# Check for duplicates in train.csv
train_duplicates = train.duplicated().sum()
print(f"Number of duplicate rows in train.csv: {train_duplicates}")

# Check for duplicates in test.csv
test_duplicates = test.duplicated().sum()
print(f"Number of duplicate rows in test.csv: {test_duplicates}")

Number of duplicate rows in train.csv: 0
Number of duplicate rows in test.csv: 0


This code checks for and counts the number of duplicate rows in the train and test DataFrames, then prints the results. This helps identify and quantify any redundant data in the training and testing datasets, which is important for ensuring data quality and accuracy in your analysis.

In [28]:
# Check for missing values in train.csv
print("Missing values in train.csv:")
print(train.isnull().sum())

# Check for missing values in test.csv
print("\nMissing values in test.csv:")
print(test.isnull().sum())

Missing values in train.csv:
headlines         0
description       0
content           0
url               0
category          0
article_length    0
full_text         0
dtype: int64

Missing values in test.csv:
headlines      0
description    0
content        0
url            0
category       0
full_text      0
dtype: int64


This code checks for and counts the number of missing values in each column of the train and test DataFrames, then prints the results. This helps identify any gaps in the data that may need to be addressed before further analysis or modeling.

In [29]:
import requests
# Function to validate URLs
def validate_url(url):
    try:
        response = requests.head(url, allow_redirects=True, timeout=5)
        return response.status_code == 200
    except requests.RequestException:
        return False

# Validate URLs in train.csv
train['url_valid'] = train['url'].apply(validate_url)
invalid_train_urls = train[~train['url_valid']].shape[0]
print(f"Number of invalid or unreachable URLs in train.csv: {invalid_train_urls}")

# Validate URLs in test.csv
test['url_valid'] = test['url'].apply(validate_url)
invalid_test_urls = test[~test['url_valid']].shape[0]
print(f"Number of invalid or unreachable URLs in test.csv: {invalid_test_urls}")

Number of invalid or unreachable URLs in train.csv: 5520
Number of invalid or unreachable URLs in test.csv: 2000


This code defines a function to check if URLs are valid and reachable by sending a HEAD request. It then applies this function to the url column in both the training and testing datasets, adds a new column indicating URL validity, and prints the number of invalid or unreachable URLs in each dataset. This helps ensure that the URLs in your data are functional and reliable.

Standardise & Group Categories

In [30]:
# Function to standardize and group categories
def standardize_category(category):
    category = category.lower()
    if category in ['business', 'economy', 'market']:
        return 'business'
    elif category in ['sports', 'cricket', 'football', 'tennis']:
        return 'sports'
    elif category in ['technology', 'tech', 'ai', 'privacy']:
        return 'technology'
    elif category in ['education', 'admissions', 'scholarships', 'policy']:
        return 'education'
    elif category in ['entertainment', 'bollywood', 'hollywood', 'cinema']:
        return 'entertainment'
    else:
        return 'miscellaneous'

# Apply the function
train['category'] = train['category'].apply(standardize_category)
test['category'] = test['category'].apply(standardize_category)

print("Category standardization and grouping completed.")

Category standardization and grouping completed.


This code defines a function to standardize and group similar categories into broader groups (e.g., 'business', 'sports', 'technology') by converting category names to lowercase and mapping them to predefined groups. It then applies this function to the category column in both the training and testing datasets, ensuring consistent and simplified category labels for analysis.

In [ ]:
# Encode labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(train['category'])

This code initializes a LabelEncoder and uses it to convert the categorical labels in the category column of the training dataset into numerical values, storing the result in the variable y. This transformation is essential for machine learning models that require numerical input.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(ngram_range=(1, 2), max_features=1000)
train_features = vectorizer.fit_transform(train['processed_text'])
test_features = vectorizer.transform(test['processed_text'])


This code initializes a CountVectorizer to convert text data into numerical feature vectors, considering unigrams and bigrams (single words and pairs of words) and limiting to the top 1,000 features. It then fits the vectorizer to the processed training text data and transforms both the training and testing text data into numerical matrices (X_train and X_test), making them suitable for machine learning models.

In [ ]:
# Split data
y = label_encoder.transform(df_train['category'])  # Already processed
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train, y, test_size=0.2, random_state=42
)

This code splits the training data into training and validation sets. It uses train_test_split to divide X_train and y into X_train_split and y_train_split for training, and X_val_split and y_val_split for validation, with 20% of the data reserved for validation. This helps evaluate the model's performance on unseen data.

# Model Training

In [ ]:
# Function to train, predict and visualize confusion matrix
def evaluate_model(model, model_name, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=model.classes_, yticklabels=model.classes_)
    plt.title(f'{model_name} - Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

    # Classification Report
    report = classification_report(y_test, y_pred, output_dict=True)
    print(f"\n{model_name} - Classification Report:")
    print(classification_report(y_test, y_pred))

    # Get macro-averaged scores
    precision = report['macro avg']['precision']
    recall = report['macro avg']['recall']
    f1 = report['macro avg']['f1-score']

    return precision, recall, f1

models = [
    (LogisticRegression(max_iter=1000), "Logistic Regression"),
    (RandomForestClassifier(n_estimators=100, random_state=42), "Random Forest"),
    (MultinomialNB(), "Multinomial Naive Bayes"),
    (KNeighborsClassifier(n_neighbors=5), "K-Nearest Neighbors")
]

# Store performance scores
model_names = []
precision_scores = []
recall_scores = []
f1_scores = []

for model, name in models:
    precision, recall, f1 = evaluate_model(model, name, X_train_split, y_train_split, X_val_split, y_val_split)
    model_names.append(name)
    precision_scores.append(precision)
    recall_scores.append(recall)
    f1_scores.append(f1)

labels = ['Precision', 'Recall', 'F1 Score']
num_vars = len(labels)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]

def add_to_radar(ax, values, label):
    values += values[:1]
    ax.plot(angles, values, label=label)
    ax.fill(angles, values, alpha=0.1)

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
for i, model in enumerate(model_names):
    scores = [precision_scores[i], recall_scores[i], f1_scores[i]]
    add_to_radar(ax, scores, model)

ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_thetagrids(np.degrees(angles[:-1]), labels)
ax.set_ylim(0.5, 1.0)  # Adjust if you're testing lower-performing models
ax.set_rgrids([0.5, 0.6, 0.7, 0.8, 0.9, 1.0], angle=90)
plt.title('Model Performance Comparison (Radar Chart)', size=15, y=1.1)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()

best_index = np.argmax(f1_scores)
final_model = models[best_index][0]
final_model.fit(X_train_split, y_train_split)

This code defines a function to train a machine learning model, make predictions, and visualize the results using a confusion matrix and classification report. It evaluates multiple models (Logistic Regression, Random Forest, Multinomial Naive Bayes, K-Nearest Neighbors), stores their performance metrics (precision, recall, F1 score), and visualizes these metrics in a radar chart. Finally, it selects the best-performing model based on the F1 score and trains it on the full training data.

Choose Best performing model for final Input

streamlit

In [ ]:
import pickle

pickle.dump(final_model, open('model.pkl', 'wb'))
pickle.dump(vectorizer, open('vectorizer.pkl', 'wb')) 
pickle.dump(label_encoder, open('label_encoder.pkl', 'wb'))

This code uses the pickle module to save the trained model, vectorizer, and label encoder to disk as binary files (model.pkl, vectorizer.pkl, and label_encoder.pkl). This allows you to easily load and reuse these components later without retraining the model or reprocessing the data.

# Final deployment

In [ ]:
import joblib

# Save the final model
joblib.dump(final_model, 'final_model.pkl')

This code snippet saves a trained machine learning model to a file named final_model.pkl using the joblib library. The joblib.dump function serializes the final_model object and writes it to the specified file, allowing the model to be easily loaded and used later.

In [ ]:
import joblib

# Load the final model
final_model = joblib.load('final_model.pkl')

# Example prediction
y_pred = final_model.predict(X_new)


This code snippet loads a pre-trained machine learning model from a file named final_model.pkl using the joblib library. It then uses this loaded model to make predictions on new data (X_new) and stores the predictions in the variable y_pred.

In [ ]:
import joblib
import numpy as np

# Load the final model
final_model = joblib.load('final_model.pkl')

# Function to make predictions
def make_predictions(model, X_new):
    return model.predict(X_new)

# Example new data
X_new = np.array([[...], [...], ...])  # Replace with your new data

# Make predictions
predictions = make_predictions(final_model, X_new)
print("Predictions:", predictions)

This code snippet loads a pre-trained machine learning model from a file named final_model.pkl using the joblib library. It defines a function to make predictions with the loaded model on new input data provided as a NumPy array. The code then uses this function to predict outcomes for the new data and prints the predictions.